# מפגש 16 — פתרון מורה עם בדיקות
פתרון P1–P5 של Student-Lab המקורית. המימוש הידני נבדק מול KNeighborsClassifier רק בבעיות שבהן קיימת הכרעת רוב חד־משמעית. המדיניות בתיקו קולות: None.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
X_train = np.array([[2.,1.],[4.,2.],[4.,4.],[6.,2.],[2.,7.]])
y_train = np.array(['B','A','A','B','A'])
queries = np.array([[2.,2.]])
original_X = X_train.copy()
original_y = y_train.copy()
assert X_train.shape == (5,2)
assert y_train.shape == (5,)
assert queries.shape == (1,2)

## 1. פונקציית מרחק עם return ולולאה
חישוב [0,0]→[3,4] הוא 5, וגם [1,2]→[4,6] הוא 5. בדיקה בסיסית של מספר מאפיינים שווה.

In [ ]:
def euclidean_distance(u, v):
    if len(u) != len(v):
        raise ValueError("Both vectors must have the same number of features")
    total = 0.0
    for j in range(len(u)):
        diff = float(u[j]) - float(v[j])
        total = total + diff ** 2
    return total ** 0.5
assert euclidean_distance([0,0],[3,4]) == 5.
assert euclidean_distance([1,2],[4,6]) == 5.
assert euclidean_distance([4,6],[1,2]) == 5.

## 2. פונקציית חיזוי עם הצמדת distance/label
אם אין מנצחת יחידה ב־mode — מחזירים None. בדיקת K מחוץ לטווח מבצעת raise ValueError.

In [ ]:
def predict_one(train_X, train_y, query, k):
    if not isinstance(k, int) or k < 1 or k > len(train_X):
        raise ValueError("K must be an integer from 1 to n_train")
    if len(train_X) != len(train_y):
        raise ValueError("Each training row must have one label")
    distances = []
    for row in train_X:
        distances.append(euclidean_distance(row, query))
    neighbors = pd.DataFrame({"distance": distances, "label": train_y})
    nearest = neighbors.sort_values("distance", kind="stable").head(k)
    modes = nearest["label"].mode().tolist()
    if len(modes) != 1:
        return None
    return modes[0]
assert predict_one(X_train,y_train,queries[0],1) == 'B'
assert predict_one(X_train,y_train,queries[0],3) == 'A'
assert predict_one(X_train,y_train,queries[0],5) == 'A'

## 3. דירוג ידני והצבעה מול הספרייה
מרחקי P1–P5 אל [2,2] הם [1,2,√8,4,5]. החיזוי במטלת התלמיד B/A/A עבור K1/K3/K5.

In [ ]:
distances=[]
for row in X_train:
    distances.append(euclidean_distance(row,queries[0]))
print('Distances:',distances)
assert np.allclose(distances,[1.,2.,np.sqrt(8),4.,5.])
for k in [1,3,5]:
    clf=KNeighborsClassifier(n_neighbors=k,weights='uniform',metric='euclidean')
    clf.fit(X_train,y_train)
    library_prediction=clf.predict(queries)[0]
    manual_prediction=predict_one(X_train,y_train,queries[0],k)
    print(k,manual_prediction,library_prediction)
    assert library_prediction == manual_prediction

## 4. שתי שאילתות חדשות במקור והשוואה מתאימה
שואלים [[3,3],[6,4]] ב־K3. במקרה השני יש שני שכנים במרחק שווה; שניהם נבחרים בין שלושת הקרובים ולכן הקולות עדיין קובעים רוב, ואילו K1 עלול להשתנות עם מדיניות הכרעה.

In [ ]:
two_queries=np.array([[3.,3.],[6.,4.]])
manual_predictions=[]
for q in two_queries:
    manual_predictions.append(predict_one(X_train,y_train,q,3))
clf_three=KNeighborsClassifier(n_neighbors=3,weights='uniform',metric='euclidean')
clf_three.fit(X_train,y_train)
library_predictions=clf_three.predict(two_queries).tolist()
print('Two manual predictions:',manual_predictions)
print('Two library predictions:',library_predictions)
assert manual_predictions == ['A','A']
assert library_predictions == manual_predictions

## 5. אותה תוצאה עם NumPy
הביטוי X−q משתמש בשידור נקודה (2,) על כל חמש השורות, ורק sum(axis=1) נותנת חמישה מרחקים.

In [ ]:
loop_distances=[]
for row in X_train:
    loop_distances.append(euclidean_distance(row,queries[0]))
vector_distances=np.sqrt(((X_train-queries[0])**2).sum(axis=1))
print('Loop:',loop_distances,'NumPy:',vector_distances)
assert vector_distances.shape == (5,)
assert np.allclose(loop_distances,vector_distances)
assert np.allclose(vector_distances,[1.,2.,np.sqrt(8),4.,5.])

## 6. תיקו, K לא תקין ומקור מוגן
מקרה A/B/C יוחזר None. K=0 או K=7 אינם חוקיים לאימון עם חמש שורות. המקור חייב להישאר ללא שינוי.

In [ ]:
tie_X=np.array([[1.,0.],[0.,1.],[-1.,0.]])
tie_y=np.array(['A','B','C'])
tie_prediction=predict_one(tie_X,tie_y,np.array([0.,0.]),3)
assert tie_prediction is None
try:
    predict_one(X_train,y_train,queries[0],7)
    assert False
except ValueError:
    print('Invalid K rejected, as expected')
assert np.array_equal(X_train,original_X)
assert np.array_equal(y_train,original_y)
assert euclidean_distance([3,4],[0,0]) == 5.

## 7. תיעוד מורה
אין תוויות אמת לשאילתות החדשות ולכן אין Accuracy. ספרייה יכולה להשתמש במדיניות שונה לתיקו קולות או למרחקים שווים בגבול K. אין להפוך פונקציית הלימוד ל־GridSearchCV; היא אינה estimator API. לפני פרויקט אמיתי מתאימים חסרים/Scaling/Encoding על Train בלבד.